# 08a — Final Evaluation, Part 1: Baselines on the Test Seasons

**Project:** Learning the Path of the Storm — 24-hour tropical cyclone track forecasting
**Author:** Yatharth Vardan · AAI-590 Capstone, University of San Diego

**Rubric section:** Model / Pipeline Analysis & Discussion (Study A final evaluation)

From the very first notebook, the 2020–2024 seasons have been locked away. No model was trained on them, no
setting was chosen with them, and no plot has shown them. So now, with every modeling decision made, it's
finally time to open them up and see how the models do on storms they have truly never seen.

As before, the final evaluation is split across two notebooks, since XGBoost and PyTorch can't
safely share a Python session on macOS. Here, in part 1, we score the four baselines from notebook 03. In
part 2 (08b), we score the neural networks and bring everything together.

Nothing is retrained or adjusted along the way. Every model is used exactly as it was saved when its own
notebook finished.

| | |
|---|---|
| **Inputs** | `data/processed/tracks_clean.csv.gz`, `models/xgb_dx.json`, `models/xgb_dy.json`, `models/xgb_config.json` |
| **Output** | `data/processed/preds_test_baselines.csv` |

### Setting things up

Here, the imports match notebook 03: scikit-learn for the linear model, XGBoost for the trees, and the
project's shared forecast pipeline from `windows.py`.

In [1]:
# Imports and paths
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

from stormtrack.data_io import load_clean, PROCESSED
from stormtrack.windows import build_samples, persistence_displacement, track_error_km, to_latlon, TARGET_COLS

MODEL_DIR = ROOT / "models"
CFG = json.loads((MODEL_DIR / "xgb_config.json").read_text())
print("XGBoost settings from notebook 03:", {k: CFG[k] for k in ("framing", "max_depth", "learning_rate", "trees")})
print("Ridge penalty from notebook 03:", CFG["ridge_alpha"])
pd.set_option("display.width", 150)

XGBoost settings from notebook 03: {'framing': 'residual', 'max_depth': 6, 'learning_rate': 0.05, 'trees': [365, 478]}
Ridge penalty from notebook 03: 0.1


## 1. Rebuilding the forecasts

First, we rebuild the forecast samples exactly as in notebook 03, with the same one-hot basin columns, and
pull out the training, validation and test sets. The test set should hold 6,745 forecasts, as counted back in
the EDA.

In [2]:
tracks = load_clean()
S = build_samples(tracks)
for b in sorted(S["BASIN"].unique()):
    S[f"basin_{b}"] = (S["BASIN"] == b).astype(float)
X_COLS = CFG["features"]

tr, va, te = (S[S["split"] == p].reset_index(drop=True) for p in ("train", "val", "test"))
print(f"train {len(tr):,}   validation {len(va):,}   test {len(te):,} forecasts from {te['SID'].nunique()} storms")
te.groupby("SEASON").agg(forecasts=("SID", "size"), storms=("SID", "nunique"))

train 65,169   validation 4,699   test 6,745 forecasts from 444 storms


,forecasts,storms
SEASON,,
2020,1215,98
2021,1421,88
2022,1239,93
2023,1530,78
2024,1340,87


So, the test set holds **6,745 forecasts from 444 storms**, matching the count from the EDA, and the five seasons are evenly sized at 1,215–1,530 forecasts each. Interestingly, 2023 has the most forecasts but the fewest storms (78), so that season's storms lasted longer on average.

## 2. Loading the models, and checking they're the same ones

Before trusting any test number, it's worth proving the models came back exactly as they were. So in the
cell below, we load the two saved XGBoost models (one per direction) and refit the ridge regression. The
ridge model was never saved, but refitting it on the same data with the same penalty gives exactly the same
model, since ridge regression has a single exact solution. Then we score both on the **validation** seasons,
where notebook 03 recorded 158.3 and 148.5 km. If those numbers come back, the models are the right ones.

In [3]:
xgb_models = []
for comp in ("dx", "dy"):
    m = xgb.XGBRegressor()
    m.load_model(MODEL_DIR / f"xgb_{comp}.json")
    xgb_models.append(m)

ridge = make_pipeline(StandardScaler(), Ridge(alpha=CFG["ridge_alpha"])).fit(tr[X_COLS].to_numpy(), tr[TARGET_COLS].to_numpy())

def forecasts(d):
    P = persistence_displacement(d)
    xg = np.column_stack([m.predict(d[X_COLS].to_numpy()) for m in xgb_models])
    return {"No motion": np.zeros((len(d), 2)), "Persistence": P,
            "Linear regression": ridge.predict(d[X_COLS].to_numpy()),
            "XGBoost": xg + (P if CFG["framing"] == "residual" else 0.0)}

base_val = pd.read_csv(PROCESSED / "preds_val_baselines.csv", keep_default_na=False, na_values=[""])
check = pd.DataFrame({m: {"notebook 03": base_val[f"err_{m.lower().replace(' ', '_')}_km"].mean(),
                          "reloaded": track_error_km(va, d).mean()}
                      for m, d in forecasts(va).items()}).T
check["difference_km"] = check["reloaded"] - check["notebook 03"]
assert check["difference_km"].abs().max() < 0.05, "reloaded models don't reproduce notebook 03"
check.round(2)

,notebook 03,reloaded,difference_km
No motion,391.15,391.15,0.0
Persistence,167.95,167.95,0.0
Linear regression,158.34,158.34,0.0
XGBoost,148.54,148.54,0.0


All four come back to within 0.00 km of notebook 03's validation numbers. So these are exactly the models that were chosen there, and any change on the test seasons has to come from the storms, not from the models.

## 3. The test seasons

Now for the moment the whole project has been building toward, at least for the baselines. The cell below
runs all four on the 2020–2024 forecasts and summarizes their errors in the same way as every earlier
notebook: the mean, median and 90th percentile, North Atlantic only, beyond 25° latitude, and skill over
persistence. The validation-season means sit alongside for comparison.

In [4]:
fc_te = forecasts(te)
err_te = {m: track_error_km(te, d) for m, d in fc_te.items()}
na, hi = te["BASIN"].eq("NA").to_numpy(), te["LAT"].abs().ge(25).to_numpy()
summary = pd.DataFrame({m: {"val_mean_km": check.loc[m, "reloaded"], "test_mean_km": e.mean(),
                            "test_median_km": np.median(e), "test_p90_km": np.percentile(e, 90),
                            "test_NA_km": e[na].mean(), "test_beyond_25deg_km": e[hi].mean()}
                        for m, e in err_te.items()}).T
summary["skill_vs_persistence_%"] = 100 * (1 - summary["test_mean_km"] / summary.loc["Persistence", "test_mean_km"])
summary.round(1)

,val_mean_km,test_mean_km,test_median_km,test_p90_km,test_NA_km,test_beyond_25deg_km,skill_vs_persistence_%
No motion,391.1,398.6,372.1,661.2,461.7,464.1,-137.5
Persistence,168.0,167.8,140.3,319.1,179.1,231.0,0.0
Linear regression,158.3,159.8,135.1,301.0,159.7,205.9,4.7
XGBoost,148.5,150.3,128.3,278.6,155.1,197.2,10.4


And here are the first test numbers. In short, validation gave a faithful preview:

- **Persistence** scores 167.8 km, almost exactly its 168.0 km on validation, so the test seasons are about as hard overall as the validation ones were.
- **XGBoost** reaches **150.3 km**, only 1.8 km worse than on validation, and keeps 10.4% skill over persistence (11.6% on validation).
- **In the North Atlantic**, XGBoost (155.1 km) beats persistence (179.1 km) by 24 km, the same pattern as before.

Still, a small drift like XGBoost's is expected. Its number of trees was picked by early stopping on the validation seasons, which flatters its validation score a little. Persistence has nothing to pick, so it shows no drift at all.

## 4. Saving for part 2

Last, we save each baseline's test error and forecast position for every test forecast, so notebook 08b
can compare the neural networks with them, forecast by forecast, without loading XGBoost.

In [5]:
out = te[["SID", "SEG_ID", "BASIN", "SEASON", "ISO_TIME", "LAT", "LON", "lat_true", "lon_true", "wind_kt"]].copy()
for m, d in fc_te.items():
    key = m.lower().replace(" ", "_")
    out[f"err_{key}_km"] = err_te[m]
    out[f"pred_lat_{key}"], out[f"pred_lon_{key}"] = to_latlon(te["LAT"], te["LON"], d[:, 0], d[:, 1])
out.to_csv(PROCESSED / "preds_test_baselines.csv", index=False)
print(f"saved data/processed/preds_test_baselines.csv ({len(out):,} rows)")

saved data/processed/preds_test_baselines.csv (6,745 rows)


## Summary

- The four baselines were reloaded unchanged and reproduce their validation scores exactly.
- On the 2020–2024 test seasons: no motion 398.6 km, persistence 167.8 km, linear regression 159.8 km, XGBoost **150.3 km** (10.4% skill over persistence).
- Every number is within 2 km of its validation counterpart, so the validation seasons were an honest guide.

Notebook 08b adds the neural networks, tests the differences, and compares with the National Hurricane Center.

## References

Chen, T., & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. In *Proceedings of the 22nd ACM
SIGKDD International Conference on Knowledge Discovery and Data Mining* (pp. 785–794).
https://doi.org/10.1145/2939672.2939785

Gahtan, J., Knapp, K. R., Schreck, C. J., Diamond, H. J., Kossin, J. P., & Kruk, M. C. (2024).
*International Best Track Archive for Climate Stewardship (IBTrACS) Project, Version 4r01*. NOAA National
Centers for Environmental Information. https://doi.org/10.25921/82ty-9e16

Neumann, C. J. (1972). *An alternate to the HURRAN tropical cyclone forecast system* (NOAA Technical
Memorandum NWS SR-62). National Weather Service.

*AI-tool disclosure: Claude (Anthropic) helped draft this notebook's code and explanations. The author
reviewed each step, ran the notebook, and checked the outputs.*